In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# 1. Clean up existing installs
!pip uninstall -y torch torchvision torchaudio

# 2. Install PyTorch 2.4.0 (Stable for P100/sm_60)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# 3. Reinstall transformers/datasets to match the new torch version
!pip install seqeval transformers datasets evaluate --upgrade

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 99.6 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 93.9 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 72.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 40.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 90.6 MB/s eta 0:00:0000:0100:01
     ━━━━

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from collections import Counter

# --------------------------------------------------
# 1. Load CoNLL-2003
# --------------------------------------------------
raw_dataset = load_dataset("lhoestq/conll2003")
print(raw_dataset)

# CoNLL-2003 label list (index matches the integer label in the dataset)
# 0:O  1:B-PER  2:I-PER  3:B-ORG  4:I-ORG  5:B-LOC  6:I-LOC  7:B-MISC  8:I-MISC
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
num_labels  = len(label_list)
label2id    = {l: i for i, l in enumerate(label_list)}
id2label    = {i: l for i, l in enumerate(label_list)}

print(f"\nNER labels ({num_labels}): {label_list}")

# --------------------------------------------------
# 2. Inspect a raw example
# --------------------------------------------------
example = raw_dataset["train"][0]
print(f"\nRaw example:")
print(f"  tokens   : {example['tokens']}")
print(f"  ner_tags : {example['ner_tags']}  -> {[label_list[t] for t in example['ner_tags']]}")

# --------------------------------------------------
# 3. Tokenizer
# --------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# --------------------------------------------------
# 4. Tokenize + align labels
# --------------------------------------------------
def tokenize_and_align_labels(examples, label_all_tokens=False):
    """
    Tokenise a batch of word-tokenised sentences and align NER labels
    to the resulting subword tokens.

    Args:
        label_all_tokens: If True, propagate the label to every subword of a
                          word (B- tags become I- tags for continuation pieces).
                          If False (default), only label the first subword and
                          set the rest to -100 so they are ignored in the loss.
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=128,
        padding=False,            # dynamic padding handled by DataCollator
        is_split_into_words=True  # input is already word-split
    )

    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                # Special token ([CLS] / [SEP] / [PAD]) → ignore
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                # First subword of a new word → assign the real label
                aligned_labels.append(word_labels[word_id])
            else:
                # Continuation subword of the same word
                if label_all_tokens:
                    lbl = word_labels[word_id]
                    # B- tag (odd indices: 1,3,5,7) → convert to I- (+1)
                    aligned_labels.append(lbl + 1 if lbl % 2 == 1 else lbl)
                else:
                    aligned_labels.append(-100)
            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


# Apply tokenization to all splits
tokenized_train      = raw_dataset["train"].map(
    tokenize_and_align_labels, batched=True,
    remove_columns=raw_dataset["train"].column_names
)
tokenized_validation = raw_dataset["validation"].map(
    tokenize_and_align_labels, batched=True,
    remove_columns=raw_dataset["validation"].column_names
)
tokenized_test       = raw_dataset["test"].map(
    tokenize_and_align_labels, batched=True,
    remove_columns=raw_dataset["test"].column_names
)

# Set PyTorch format
for ds in [tokenized_train, tokenized_validation, tokenized_test]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# --------------------------------------------------
# 5. Sanity checks
# --------------------------------------------------
print(f"\nDataset sizes:")
print(f"  Train:      {len(tokenized_train):,}")
print(f"  Validation: {len(tokenized_validation):,}")
print(f"  Test:       {len(tokenized_test):,}")

sample = tokenized_train[0]
print(f"\nFirst training example (tokenized):")
print(f"  input_ids shape : {sample['input_ids'].shape}")
print(f"  attention_mask  : {sample['attention_mask']}")
print(f"  labels          : {sample['labels']}")
print(f"  decoded tokens  : {tokenizer.convert_ids_to_tokens(sample['input_ids'].tolist())}")
print(f"  label names     : {[id2label[l.item()] if l.item() != -100 else 'IGN' for l in sample['labels']]}")

# Label distribution (excluding -100)
flat_labels = [l.item() for ex in tokenized_train for l in ex["labels"] if l.item() != -100]
print(f"\nLabel distribution in training set:")
for label_id, count in sorted(Counter(flat_labels).items()):
    print(f"  {id2label[label_id]:8s} ({label_id}): {count:,}")

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]


Dataset sizes:
  Train:      14,041
  Validation: 3,250
  Test:       3,453

First training example (tokenized):
  input_ids shape : torch.Size([11])
  attention_mask  : tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
  labels          : tensor([-100,    3,    0,    7,    0,    0,    0,    7,    0,    0, -100])
  decoded tokens  : ['[CLS]', 'eu', 'rejects', 'german', 'call', 'to', 'boycott', 'british', 'lamb', '.', '[SEP]']
  label names     : ['IGN', 'B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O', 'IGN']

Label distribution in training set:
  O        (0): 169,554
  B-PER    (1): 6,600
  I-PER    (2): 4,528
  B-ORG    (3): 6,321
  I-ORG    (4): 3,704
  B-LOC    (5): 7,140
  I-LOC    (6): 1,157
  B-MISC   (7): 3,438
  I-MISC   (8): 1,155


In [7]:
import torch
import torch.nn as nn
import inspect
from transformers import BertConfig, BertForTokenClassification


class BertForNER(nn.Module):
    def __init__(self, vocab_size=30522, num_labels=4):
        super().__init__()

        # === BERT config  ===
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=768,
            num_hidden_layers=12,
            num_attention_heads=12,
            intermediate_size=3072,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1,
            max_position_embeddings=512,
            type_vocab_size=2,
            num_labels=num_labels
        )

        # KEY CHANGE vs AG News:
        # BertForTokenClassification puts a linear head on EVERY token,
        # not just [CLS]. It also uses ignore_index=-100 internally,
        # so padding / subword-continuation labels are automatically ignored.
        self.model = BertForTokenClassification(config)

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        Forward pass for NER (token classification).
        Returns logits (B, T, num_labels) + loss (if labels provided).
        Labels shape: (B, T), with -100 for ignored positions.
        """
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs.logits, outputs.loss

    def configure_optimizers(self, weight_decay, learning_rate, device="cuda", verbose=False):
        """
        AdamW optimizer with weight decay only on weight matrices.
        """
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params   = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

        optim_groups = [
            {"params": decay_params,   "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]

        fused_available = "fused" in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and "cuda" in device

        if verbose:
            num_decay   = sum(p.numel() for p in decay_params)
            num_nodecay = sum(p.numel() for p in nodecay_params)
            print(f"decay params:   {len(decay_params)} tensors, {num_decay/1e6:.2f}M params")
            print(f"nodecay params: {len(nodecay_params)} tensors, {num_nodecay/1e6:.2f}M params")
            print(f"Using fused AdamW: {use_fused}")

        optimizer = torch.optim.AdamW(
            optim_groups,
            lr=learning_rate,
            betas=(0.9, 0.999),
            eps=1e-8,
            fused=use_fused
        )
        return optimizer


# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# Set random seeds for reproducibility
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

# Initialize the model — 9 labels for CoNLL-2003 BIO tags:
# O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-MISC, I-MISC
model = BertForNER(num_labels=9)
model = model.to(device)

total_params = sum(param.numel() for param in model.parameters())
print(f"Total number of parameters: {total_params/1e6:.2f}M")

# TF32
torch.set_float32_matmul_precision('high')

Total number of parameters: 108.90M


In [8]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import f1_score as seq_f1
from tqdm import tqdm
import time
import math
from datetime import datetime, timezone

# --------------------------------------------------
# 1. DataLoaders
# --------------------------------------------------
batch_size = 64

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    max_length=128,
    padding="max_length",
    label_pad_token_id=-100
)

train_dataloader = DataLoader(
    tokenized_train,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator
)
val_dataloader = DataLoader(
    tokenized_validation,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=data_collator
)
test_dataloader = DataLoader(
    tokenized_test,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=data_collator
)

print(f"Training batches:   {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches:       {len(test_dataloader)}")

# --------------------------------------------------
# 2. Training Hyperparameters
# --------------------------------------------------
num_epochs    = 10
max_steps     = num_epochs * len(train_dataloader)
grad_clip     = 1.0
eval_interval = 200
log_interval  = 50

max_lr       = 1e-4
min_lr       = 1e-5
warmup_steps = int(0.05 * max_steps)
plateau      = int(0.25 * max_steps)

def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it < plateau:
        return max_lr
    if it >= max_steps:
        return min_lr
    decay_ratio = (it - plateau) / (max_steps - plateau)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

optimizer = model.configure_optimizers(
    weight_decay=0.01,
    learning_rate=max_lr,
    device=device,
    verbose=True
)

# History tracking
train_losses  = []
val_losses    = []
val_f1s       = []
steps_history = []


# --------------------------------------------------
# 3. seqeval helper
# --------------------------------------------------
def convert_predictions_to_seqeval(logits, labels):
    """
    Convert model logits + label tensors to seqeval-compatible format.
    Filters out all -100 positions (special tokens, padding, subword continuations).

    Args:
        logits : (B, T, num_labels) — float tensor on CPU
        labels : (B, T)             — long tensor on CPU, -100 for ignored positions

    Returns:
        true_labels : list[list[str]]
        pred_labels : list[list[str]]
    """
    preds = torch.argmax(logits, dim=-1)  # (B, T)
    true_labels, pred_labels = [], []

    for pred_seq, label_seq in zip(preds, labels):
        true_seq, pred_seq_out = [], []
        for p, l in zip(pred_seq.tolist(), label_seq.tolist()):
            if l != -100:
                true_seq.append(id2label[l])
                pred_seq_out.append(id2label[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_out)

    return true_labels, pred_labels


# --------------------------------------------------
# 4. Evaluation (training-time — loss + entity F1 only)
# --------------------------------------------------
def evaluate(dataloader, split_name):
    """Evaluation function for NER with entity-level F1."""
    model.eval()
    total_loss = 0
    all_true   = []
    all_pred   = []

    progress_bar = tqdm(dataloader, desc=f"Evaluating {split_name}",
                        leave=True, position=0, ncols=80,
                        bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

    with torch.no_grad():
        for batch in progress_bar:
            input_ids      = batch["input_ids"].to(device)
            labels         = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device).bool()

            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                logits, loss = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            total_loss += loss.item()

            true_b, pred_b = convert_predictions_to_seqeval(
                logits.detach().cpu(),
                labels.detach().cpu()
            )
            all_true.extend(true_b)
            all_pred.extend(pred_b)

            progress_bar.set_postfix(loss=f"{loss.item():.4f}", refresh=False)

    avg_loss = total_loss / len(dataloader)
    f1       = seq_f1(all_true, all_pred)

    print(f"\n{split_name} Results | Loss: {avg_loss:.4f} | Entity F1: {f1:.4f}")

    return {"loss": avg_loss, "f1": f1}


# --------------------------------------------------
# 5. Plotting
# --------------------------------------------------
def plot_training_history():
    """Plot training and validation metrics."""
    plt.figure(figsize=(12, 7))

    # Plot losses
    plt.subplot(2, 1, 1)
    plt.plot(steps_history, train_losses, label='Train Loss')
    plt.plot(steps_history, val_losses,   label='Val Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)

    # Plot Entity F1
    plt.subplot(2, 1, 2)
    plt.plot(steps_history, val_f1s, label='Val Entity F1', color='green')
    plt.xlabel('Steps')
    plt.ylabel('Entity F1')
    plt.title('Validation Entity-level F1')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('ner_training_history.png')
    plt.close()
    print("Training history plot saved to 'ner_training_history.png'")


# --------------------------------------------------
# 6. Training Loop
# --------------------------------------------------
def train():
    global_step = 0
    best_val_f1 = 0.0
    start_time  = time.time()

    global train_losses, val_losses, val_f1s, steps_history

    for epoch in range(num_epochs):
        print(f"\n{'='*60}")
        print(f"Starting epoch {epoch+1}/{num_epochs}")
        print(f"{'='*60}")
        model.train()
        epoch_losses = []

        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}",
                            leave=True, position=0, ncols=80,
                            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]')

        for batch in progress_bar:
            if global_step >= max_steps:
                break

            t0 = time.time()

            # Get batch data
            input_ids      = batch["input_ids"].to(device)
            labels         = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device).bool()

            optimizer.zero_grad()

            # Forward pass
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                logits, loss = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            # Backward pass
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            # Update learning rate
            lr = get_lr(global_step)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

            optimizer.step()

            current_loss = loss.item()
            epoch_losses.append(current_loss)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            t1 = time.time()
            dt = t1 - t0
            tokens_processed = input_ids.size(0) * input_ids.size(1)
            tokens_per_sec   = tokens_processed / dt

            progress_bar.set_postfix({
                'loss':  f'{current_loss:.4f}',
                'lr':    f'{lr:.2e}',
                'tok/s': f'{tokens_per_sec:.0f}'
            })

            if global_step % log_interval == 0:
                print(f'\nstep {global_step:6d} | loss: {current_loss:.6f} | lr: {lr:.4e} | '
                      f'dt: {dt*1000:.2f}ms | norm: {norm:.4f} | tok/sec: {tokens_per_sec:.2f}')

            # Evaluation
            if global_step > 0 and global_step % eval_interval == 0:
                print(f"\n{'='*60}")
                print(f"Evaluating at step {global_step}...")
                print(f"{'='*60}")
                val_metrics = evaluate(val_dataloader, "Validation")

                val_f1 = val_metrics["f1"]

                # Track metrics
                avg_train_loss = sum(epoch_losses[-100:]) / min(len(epoch_losses), 100)
                train_losses.append(avg_train_loss)
                val_losses.append(val_metrics["loss"])
                val_f1s.append(val_f1)
                steps_history.append(global_step)

                # Save best model
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    torch.save(model.state_dict(), "ner_best_model.pt")
                    print(f"✓ New best entity F1: {best_val_f1:.4f}")

                plot_training_history()
                model.train()
                print("")

            global_step += 1

        epoch_loss = sum(epoch_losses) / len(epoch_losses)
        print(f"\nEpoch {epoch+1} completed | Average loss: {epoch_loss:.6f}")

    # Save final model
    torch.save(model.state_dict(), "ner_final_model.pt")
    end_time    = time.time()
    elapsed     = end_time - start_time
    elapsed_str = datetime.fromtimestamp(elapsed, tz=timezone.utc).strftime("%H:%M:%S")

    print(f"\n{'='*60}")
    print("Training Summary")
    print(f"{'='*60}")
    print(f"Training completed in {elapsed:.2f} seconds ({elapsed_str})")
    print(f"Best entity F1: {best_val_f1:.4f}")
    print("Final model saved to 'ner_final_model.pt'")
    print("Best model saved to 'ner_best_model.pt'")



# --------------------------------------------------
# Run
# --------------------------------------------------
train()

Training batches:   220
Validation batches: 51
Test batches:       54
decay params:   76 tensors, 108.78M params
nodecay params: 123 tensors, 0.12M params
Using fused AdamW: True

Starting epoch 1/10


Epoch 1:   0%| | 1/220 [00:00<03:08,  1.16it/s, loss=2.3282, lr=9.09e-07, tok/s=


step      0 | loss: 2.328230 | lr: 9.0909e-07 | dt: 843.41ms | norm: 33.7032 | tok/sec: 9712.94


Epoch 1:  23%|▏| 51/220 [00:42<02:21,  1.20it/s, loss=0.5254, lr=4.64e-05, tok/s


step     50 | loss: 0.525449 | lr: 4.6364e-05 | dt: 819.59ms | norm: 3.1695 | tok/sec: 9995.23


...


step   2100 | loss: 0.001673 | lr: 1.0813e-05 | dt: 820.25ms | norm: 0.3353 | tok/sec: 9987.19


Epoch 10:  78%|▊| 171/220 [02:38<00:40,  1.20it/s, loss=0.0014, lr=1.02e-05, tok


step   2150 | loss: 0.001365 | lr: 1.0204e-05 | dt: 819.64ms | norm: 0.2498 | tok/sec: 9994.61


Epoch 10: 100%|█| 220/220 [03:18<00:00,  1.11it/s, loss=0.0028, lr=1.00e-05, tok



Epoch 10 completed | Average loss: 0.005219

Training Summary
Training completed in 1997.52 seconds (00:33:17)
Best entity F1: 0.7667
Final model saved to 'ner_final_model.pt'
Best model saved to 'ner_best_model.pt'


In [9]:
# ============================================================================
# FINAL EVALUATION
# ============================================================================

def final_evaluation(model, dataloader, split_name, device=None):
    """
    Final NER evaluation using seqeval — the standard for CoNLL-2003.

    Correctness is measured at the entity-span level, not per token:
    a predicted B-PER I-PER span only counts as correct if both tokens
    match exactly. seqeval's classification_report handles this and gives
    per-entity-type precision / recall / F1, which is the proper NER metric.
    """
    if device is None:
        device = next(model.parameters()).device

    from seqeval.metrics import classification_report as seq_report

    model.eval()
    all_true     = []
    all_pred     = []
    total_loss   = 0
    num_examples = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating {split_name}", leave=False):
            input_ids      = batch["input_ids"].to(device)
            labels         = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device).bool()

            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                logits, loss = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            true_b, pred_b = convert_predictions_to_seqeval(
                logits.detach().cpu(),
                labels.detach().cpu()
            )
            all_true.extend(true_b)
            all_pred.extend(pred_b)

            total_loss   += loss.item() * input_ids.size(0)
            num_examples += input_ids.size(0)

    avg_loss  = total_loss / num_examples
    f1_micro  = seq_f1(all_true, all_pred, average="micro")
    f1_macro  = seq_f1(all_true, all_pred, average="macro")
    f1_weighted = seq_f1(all_true, all_pred, average="weighted")
    report    = seq_report(all_true, all_pred, digits=4)

    print(f"\n{'='*60}")
    print(f"{split_name} EVALUATION RESULTS".center(60))
    print(f"{'='*60}")
    print(f"Loss:                {avg_loss:.4f}")
    print(f"Entity F1 (micro):   {f1_micro:.4f}")
    print(f"Entity F1 (macro):   {f1_macro:.4f}")
    print(f"Entity F1 (weighted):{f1_weighted:.4f}")
    print(f"\n{'='*60}")
    print("SEQEVAL CLASSIFICATION REPORT".center(60))
    print(f"{'='*60}")
    print(report)

    return {
        "loss":        avg_loss,
        "f1_micro":    f1_micro,
        "f1_macro":    f1_macro,
        "f1_weighted": f1_weighted,
    }


# ============================================================================
# MAIN EVALUATION SCRIPT
# ============================================================================

print("Loading best model...")
model.load_state_dict(torch.load("ner_best_model.pt", map_location=device))
model = model.to(device)
model.eval()
print("Model loaded successfully!\n")

print("\nStarting final evaluation on test set...")
test_metrics = final_evaluation(
    model=model,
    dataloader=test_dataloader,
    split_name="Test Set",
    device=device
)

print("\n" + "="*60)
print("FINAL TEST SET SUMMARY".center(60))
print("="*60)
print(f"Loss:                {test_metrics['loss']:.4f}")
print(f"Entity F1 (micro):   {test_metrics['f1_micro']:.4f}")
print(f"Entity F1 (macro):   {test_metrics['f1_macro']:.4f}")
print(f"Entity F1 (weighted):{test_metrics['f1_weighted']:.4f}")
print("="*60)

Loading best model...


/tmp/ipykernel_55/1120521351.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("ner_best_model.pt", map_location=device))


Model loaded successfully!


Starting final evaluation on test set...



                Test Set EVALUATION RESULTS                 
Loss:                0.3747
Entity F1 (micro):   0.6959
Entity F1 (macro):   0.6871
Entity F1 (weighted):0.6949

               SEQEVAL CLASSIFICATION REPORT                
              precision    recall  f1-score   support

         LOC     0.7896    0.8284    0.8085      1667
        MISC     0.6230    0.6638    0.6428       702
         ORG     0.6428    0.6261    0.6343      1661
         PER     0.6715    0.6541    0.6627      1616

   micro avg     0.6933    0.6985    0.6959      5646
   macro avg     0.6817    0.6931    0.6871      5646
weighted avg     0.6919    0.6985    0.6949      5646


                   FINAL TEST SET SUMMARY                   
Loss:                0.3747
Entity F1 (micro):   0.6959
Entity F1 (macro):   0.6871
Entity F1 (weighted):0.6949
